# Faruq-v3 DC2 raw RGB local-stream resolution screen

Mechanistic broad-search screen for the raw-crop local stream of DC2. GT object boxes from train/validation are cropped directly from the original RGB image, then resized to the predeclared 32/64/128/224 resolutions and classified by the same MobileNetV3-Small local backbone. This is not detector mAP and not yet full end-to-end DC2/MSFA. Test is never restored or opened.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, shutil, subprocess, sys, tarfile, time
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/dc2-raw-crop-screening'
os.chdir('/content')
if REPO.exists():
    shutil.rmtree(REPO)
clone = ['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)]
for attempt in range(1, 4):
    result = subprocess.run(clone)
    if result.returncode == 0:
        break
    if REPO.exists():
        shutil.rmtree(REPO)
    if attempt == 3:
        raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
for module_name in list(sys.modules):
    if module_name == 'coffee_detector' or module_name.startswith('coffee_detector.'):
        sys.modules.pop(module_name, None)
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)
print('REPO:', REPO)

In [ ]:
import torch
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root

assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=(
    'bundles/faruq-development-v3-grouped.tar',
))
ARCHIVE = require_project_artifact(PROJECT_ROOT, 'bundles/faruq-development-v3-grouped.tar')
DATA_ROOT = Path('/content/faruq-development-v3-grouped')
GROUPED_SUMMARY = DATA_ROOT / 'faruq_grouped_summary.json'
OUTPUT_ROOT = PROJECT_ROOT / 'experiments/faruq-v3-dc2-raw-crop-resolution-search-v1'
if not GROUPED_SUMMARY.is_file():
    with tarfile.open(ARCHIVE, 'r') as archive:
        archive.extractall('/content', filter='data')
assert GROUPED_SUMMARY.is_file(), GROUPED_SUMMARY
assert not (DATA_ROOT / 'test').exists(), 'Test tidak boleh tersedia.'
print('GPU       :', torch.cuda.get_device_name(0))
print('PROJECT   :', PROJECT_ROOT)
print('DATA      :', DATA_ROOT)
print('OUTPUT    :', OUTPUT_ROOT)

In [ ]:
command = [sys.executable, '-m', 'pytest', '-q', 'tests/test_dc2_crop.py']
print('STATIC CHECK:', ' '.join(command))
subprocess.run(command, cwd=REPO, check=True)
print('PASS: raw crop-before-resize, resolution set, classifier contract, and metrics verified.')

In [ ]:
command = [
    sys.executable, '-u', '-m', 'coffee_detector.experiments.run_faruq_v3_dc2_crop_screening',
    '--data-root', str(DATA_ROOT),
    '--grouped-summary', str(GROUPED_SUMMARY),
    '--output-root', str(OUTPUT_ROOT),
    '--seed', '42', '--device', '0', '--epochs', '20', '--batch-size', '64', '--workers', '2',
    '--authorize-training',
]
print('MENJALANKAN:', ' '.join(command), flush=True)
process = subprocess.run(command, cwd=REPO, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(process.stdout, end='', flush=True)
if process.returncode != 0:
    raise RuntimeError(f'DC2 crop screen gagal dengan return code {process.returncode}; traceback lengkap tercetak di atas.')

In [ ]:
import json, pandas as pd
from IPython.display import display

SUMMARY = OUTPUT_ROOT / 'dc2_raw_crop_resolution_screening.json'
assert SUMMARY.is_file(), f'DC2 crop screen belum selesai: {SUMMARY}'
result = json.loads(SUMMARY.read_text(encoding='utf-8'))
assert result['evaluation_split'] == 'val_gt_crops'
assert result['test_images_accessed'] is False
rows = []
for resolution, arm in result['results'].items():
    rows.append({'resolution': int(resolution), **{k: v for k, v in arm['metrics'].items() if k != 'per_class'}})
frame = pd.DataFrame(rows).sort_values('resolution')
display(frame.style.format({'accuracy': '{:.2%}', 'macro_f1': '{:.2%}', 'bottom3_f1': '{:.2%}', 'worst_f1': '{:.2%}'}))
print('BEST RESOLUTION :', result['best_resolution'])
print('DELTA vs 32     :', result['deltas_best_vs_32'])
print('CRITERIA        :', result['criteria'])
print('DECISION        :', result['decision'])
print('NEXT            :', result['next_action'])
print('SUMMARY         :', SUMMARY)
print('Catatan: ini klasifikasi GT crop, bukan detector mAP dan belum full DC2.')